# LightGBM Training

This notebook is the training pipeline for the approved target definition. It relies on reusable functions and does not duplicate the full ML logic in notebook cells.

The target definition used here is:
- future KEV appearance within a future time window
- time-based leakage-safe train/validation/test splits

In [ ]:
# Setup cell: install missing dependencies and configure the project root.
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()

if "google.colab" in sys.modules:
    repo_url = os.getenv("GITHUB_REPO_URL")
    if repo_url:
        !git clone {repo_url} /content/cyberguard-ai
        %cd /content/cyberguard-ai
        REPO_ROOT = Path.cwd()
    else:
        print("Set GITHUB_REPO_URL before running this notebook in Colab.")

if not (REPO_ROOT / "data" / "processed" / "risk_features.csv").exists():
    for candidate in [Path("."), Path("/content/cyberguard-ai"), Path("/workspace"), Path("/content")]:
        if (candidate / "data" / "processed" / "risk_features.csv").exists():
            REPO_ROOT = candidate
            break

os.chdir(REPO_ROOT)
print(f"Project root: {REPO_ROOT}")

!python -m pip install -q pandas numpy scikit-learn lightgbm matplotlib plotly

from src.ml.colab_setup import project_setup
PROJECT_ROOT = project_setup()
print(f"Setup root: {PROJECT_ROOT}")

In [ ]:
# Import reusable ML training functions.
from src.ml.train_lightgbm import train_and_evaluate

results = train_and_evaluate()
print(results)

## Training workflow

1. Load the approved dataset from data/processed.
2. Build the future-target label.
3. Use leakage-safe preprocessing.
4. Split by time or randomly only after confirming no leakage risk.
5. Train LightGBM.
6. Save model and outputs.
7. Evaluate carefully before claiming performance.